# 03: 量子テレポーテーション — 回路の実装

ノート `03_quantum_teleportation.md` の「プロトコルの回路表現」以降を Qiskit で実装し、シミュレーションで検証する。

**内容:**
1. ベル対の生成回路
2. 量子テレポーテーション回路（全プロトコル）
3. シミュレーションによる検証

> **記法の注意:** ノートと Qiskit ではインデックスの始まり（1 vs 0）やケット表記のビット順が異なる。詳しくは [01_qubit_ordering.ipynb](01_qubit_ordering.ipynb) を参照。物理的な結果には影響しない。

In [30]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector
import numpy as np

**インポートの解説:**

- `QuantumCircuit`: 量子回路を構築するための基本クラス。量子ビットと古典ビットを持ち、ゲートや測定を追加していく。
- `QuantumRegister`: 名前付きの量子ビットのグループ。複数のレジスタを区別したいときに使う。
- `ClassicalRegister`: 測定結果を格納する古典ビットのグループ。
- `Statevector`: 量子回路の状態ベクトルを計算・操作するクラス。シミュレーションに使用する。
- `numpy`: 数値計算ライブラリ。振幅の計算や検証に使用する。

### 初期状態

回路の各量子ビットの初期値：

| 回路の量子ビット | 初期値 | 役割 |
|---|---|---|
| $q_0$ | $\vert\psi\rangle = \alpha\vert 0\rangle + \beta\vert 1\rangle$ | 転送したい状態 |
| $q_1$ | $\vert 0\rangle$ | ベル対の Alice 側 |
| $q_2$ | $\vert 0\rangle$ | ベル対の Bob 側 |

Qiskit のリトルエンディアン規約により、全体の初期状態をケット表記すると：

$$
\vert q_2\, q_1\, q_0\rangle = \vert 0\rangle \otimes \vert 0\rangle \otimes \vert\psi\rangle = \vert 0, 0, \psi\rangle
$$

$q_0$（転送したい状態）が右端に来ることに注意。例えば $\vert\psi\rangle = \vert 0\rangle$ なら全体は $\vert 000\rangle$ である。

## 1. ベル対の生成

![ベル対生成回路](../images/03_bell_pair.png)

$$
\vert 00\rangle \xrightarrow{H \otimes I} \frac{\vert 0\rangle + \vert 1\rangle}{\sqrt{2}} \vert 0\rangle \xrightarrow{\text{CNOT}_{01}} \frac{\vert 00\rangle + \vert 11\rangle}{\sqrt{2}} = \vert\Phi^+\rangle
$$

まずベル対の生成だけを取り出して確認する。ここでは $q_0, q_1$ の2量子ビットで構成する。

（セクション2のテレポーテーション全体の回路では、$q_0$ は転送したい状態に使うため、ベル対は $q_1, q_2$ に生成する。）

In [31]:
# ベル対生成回路
bell = QuantumCircuit(2, name="Bell pair")
bell.h(0)
bell.cx(0, 1)

print(f"ベル対生成回路: {bell.num_qubits} 量子ビット, ゲート数 {bell.size()}")

ベル対生成回路: 2 量子ビット, ゲート数 2


**コードの解説:**

- `QuantumCircuit(2, name="Bell pair")`: 2量子ビットの回路を作成する。第1引数 `2` は量子ビット数。`name` は回路の名前。
- `bell.h(0)`: $q_0$ にアダマールゲート $H$ を適用する。
- `bell.cx(0, 1)`: $\text{CNOT}_{01}$ を適用する。第1引数 `0` が制御ビット（$q_0$）、第2引数 `1` が標的ビット（$q_1$）。

In [32]:
# 状態ベクトルを確認: |00⟩ + |11⟩ / √2
sv_bell = Statevector.from_instruction(bell)
print("ベル状態 |Φ+⟩:")
print(sv_bell.draw("text"))
print()
print("各基底の振幅:")
for i, amp in enumerate(sv_bell):
    if abs(amp) > 1e-10:
        print(f"  |{i:02b}⟩: {amp:.4f}")

ベル状態 |Φ+⟩:
[0.70710678+0.j,0.        +0.j,0.        +0.j,0.70710678+0.j]

各基底の振幅:
  |00⟩: 0.7071+0.0000j
  |11⟩: 0.7071+0.0000j


**コードの解説:**

- `Statevector.from_instruction(bell)`: 回路 `bell` を $\vert 00\rangle$ から実行したときの最終状態ベクトルを計算する。実機ではなくシミュレーションで状態を直接取得する方法。
- `sv.draw("text")`: 状態ベクトルをテキスト形式で表示する。
- `enumerate(sv)`: 状態ベクトルの各振幅をインデックス付きで取得する。インデックス `i` は計算基底 $\vert i\rangle$ に対応する（Qiskit のリトルエンディアン規約）。
- 出力中の `j` は虚数単位（数学・物理の $i$）。Python/NumPy では電気工学の慣習に従い `j` を使う。例えば `0.7071+0.0000j` は $0.7071 + 0i$ を意味する。

## 2. 量子テレポーテーション回路（全体）

セクション1ではベル対の生成だけを取り出して確認した。ここでは、ベル対の生成を含むテレポーテーションの全プロトコルを1つの回路にまとめる。

![テレポーテーション回路](../images/03_teleportation.png)

| Qiskit | 所持者 | 役割 |
|---|---|---|
| $q_0$ | Alice | 転送したい状態 $\vert\psi\rangle$ |
| $q_1$ | Alice | ベル対の Alice 側 |
| $q_2$ | Bob | ベル対の Bob 側 |

In [33]:
def teleportation_circuit(init_gate=None):
    """量子テレポーテーション回路を構築する。
    
    init_gate: q[0] に適用する初期化ゲート（省略時は |0⟩ のまま）
    """
    q = QuantumRegister(3, "q")
    c = ClassicalRegister(2, "c")  # Alice の測定結果
    qc = QuantumCircuit(q, c)

    # --- 転送したい状態の準備 ---
    if init_gate is not None:
        qc.append(init_gate, [q[0]])
    qc.barrier(label="init")

    # --- 準備フェーズ: ベル対生成 (q[1], q[2]) ---
    qc.h(q[1])
    qc.cx(q[1], q[2])
    qc.barrier(label="Bell pair")

    # --- Alice の操作 ---
    qc.cx(q[0], q[1])      # CNOT: q[0](制御), q[1](標的)
    qc.h(q[0])              # アダマール: q[0]
    qc.barrier(label="Alice")

    # --- 測定 ---
    qc.measure(q[0], c[0])  # c[0] = q[0] の測定結果
    qc.measure(q[1], c[1])  # c[1] = q[1] の測定結果
    qc.barrier(label="measure")

    # --- Bob の補正 ---
    with qc.if_test((c[1], 1)):  # c[1]=1 なら X を適用
        qc.x(q[2])
    with qc.if_test((c[0], 1)):  # c[0]=1 なら Z を適用
        qc.z(q[2])

    return qc

# 回路を構築（検証用に使う）
qc_teleport = teleportation_circuit()
print(f"テレポーテーション回路: {qc_teleport.num_qubits} 量子ビット, {qc_teleport.num_clbits} 古典ビット")

テレポーテーション回路: 3 量子ビット, 2 古典ビット


**コードの解説:**

- `QuantumRegister(3, "q")`: 3量子ビットのレジスタを `"q"` という名前で作成する。`q[0]`, `q[1]`, `q[2]` でアクセスする。
- `ClassicalRegister(2, "c")`: 2古典ビットのレジスタ。Alice の測定結果 $c_0, c_1$ を格納する。
- `QuantumCircuit(q, c)`: レジスタを指定して回路を作成する。`QuantumCircuit(3)` と違い、レジスタに名前が付くので回路図が読みやすくなる。
- `qc.append(init_gate, [q[0]])`: 任意のゲートを指定した量子ビットに適用する。
- `qc.barrier(label="...")`: 回路図に区切り線を入れる。回路の論理的なフェーズを視覚的に分ける。最適化の境界にもなる。
- `qc.cx(q[0], q[1])`: $\text{CNOT}_{01}$（$q_0$ 制御、$q_1$ 標的）。
- `qc.measure(q[0], c[0])`: $q_0$ を測定し、結果を古典ビット `c[0]` に格納する。
- `qc.if_test((c[1], 1))`: 古典ビット `c[1]` が 1 のときだけ、ブロック内のゲートを実行する。Bob の条件付き補正に使用。

## 3. シミュレーションによる検証

Qiskit の `c_if` は条件付きゲートだが、状態ベクトルシミュレータでは測定後の条件分岐を直接扱えない。
そこで、測定結果の4パターンそれぞれについて手動で補正を適用し、Bob の状態が元の $|\psi\rangle$ と一致するか検証する。

### 3.1 検証方法

測定なし・補正なしの回路で $|\Psi_2\rangle$ を構築し、Alice の測定結果ごとに Bob の状態を取り出す。

![検証回路](../images/03_verify_circuit.png)

In [34]:
from qiskit.circuit.library import UGate

def verify_teleportation(alpha, beta, label=""):
    """テレポーテーションを状態ベクトルレベルで検証する。
    
    転送したい状態: α|0⟩ + β|1⟩
    """
    # 転送したい元の状態
    psi = np.array([alpha, beta], dtype=complex)
    psi = psi / np.linalg.norm(psi)  # 正規化
    
    # 測定・補正なしの回路を構築（|Ψ₂⟩ を得る）
    qc = QuantumCircuit(3)
    
    # q[0] を |ψ⟩ に初期化
    qc.initialize(psi, 0)
    
    # ベル対生成: q[1], q[2]
    qc.h(1)
    qc.cx(1, 2)
    
    # Alice の操作
    qc.cx(0, 1)
    qc.h(0)
    
    # 状態ベクトルを取得
    # 3量子ビットなので 2^3 = 8 個の振幅を持つ
    sv = Statevector.from_instruction(qc)
    
    # 補正ゲート行列
    I_gate = np.eye(2)
    X_gate = np.array([[0, 1], [1, 0]])
    Z_gate = np.array([[1, 0], [0, -1]])
    
    # Alice の測定結果 (q_0, q_1) の4通りに対応する補正ゲート
    correction_names = {"00": "I", "01": "X", "10": "Z", "11": "ZX"}
    corrections = {
        "00": I_gate,
        "01": X_gate,
        "10": Z_gate,
        "11": Z_gate @ X_gate,
    }
    
    print(f"{'='*50}")
    print(f"転送する状態: {psi[0]:.4f}|0⟩ + {psi[1]:.4f}|1⟩  {label}")
    print(f"{'='*50}")
    
    all_match = True
    for bits, correction in corrections.items():
        m0, m1 = int(bits[0]), int(bits[1])  # Alice の測定結果: q_0=m0, q_1=m1
        
        # Bob の q_2 の状態を状態ベクトルから取り出す。
        # Qiskit のリトルエンディアン規約: index = q0 + q1*2 + q2*4
        # Alice の測定結果 (q0=m0, q1=m1) を固定したとき、
        # Bob の q2 が 0 か 1 かで2つのインデックスが決まる。
        idx0 = m0 + m1 * 2 + 0 * 4  # Bob の q2=0 に対応するインデックス
        idx1 = m0 + m1 * 2 + 1 * 4  # Bob の q2=1 に対応するインデックス
        # この2つの振幅が Bob の量子ビットの状態ベクトル [⟨0|bob⟩, ⟨1|bob⟩]
        bob_state = np.array([sv[idx0], sv[idx1]])
        
        # 正規化（各測定結果の確率は 1/4）
        norm = np.linalg.norm(bob_state)
        if norm > 1e-10:
            bob_state = bob_state / norm
        
        # 補正を適用
        corrected = correction @ bob_state
        
        # グローバル位相を除いて比較
        if abs(corrected[0]) > 1e-10:
            phase = psi[0] / corrected[0]
        elif abs(corrected[1]) > 1e-10:
            phase = psi[1] / corrected[1]
        else:
            phase = 1
        corrected_aligned = corrected * phase
        
        match = np.allclose(corrected_aligned, psi, atol=1e-6)
        all_match = all_match and match
        status = "✓" if match else "✗"
        
        print(f"\n  Alice の測定: |{bits}⟩")
        print(f"  Bob の状態（補正前）: {bob_state[0]:.4f}|0⟩ + {bob_state[1]:.4f}|1⟩")
        print(f"  補正ゲート: {correction_names[bits]}")
        print(f"  Bob の状態（補正後）: {corrected[0]:.4f}|0⟩ + {corrected[1]:.4f}|1⟩  {status}")
    
    print(f"\n  全パターンで |ψ⟩ を復元: {'成功' if all_match else '失敗'}")
    return all_match

**コードの解説:**

- `qc.initialize(psi, 0)`: $q_0$ を指定した状態ベクトル `psi` = $[\alpha, \beta]$ に初期化する。内部的には $\vert 0\rangle$ から目的の状態に変換するゲート列を自動生成する。
- `Statevector.from_instruction(qc)`: 測定なしの回路から状態ベクトルを取得する。測定を入れると状態が崩壊するため、検証では測定なしの回路を使う。
- `np.array(sv)`: 状態ベクトルを NumPy 配列として取得する。インデックスは Qiskit のリトルエンディアン規約で `index = q0 + q1*2 + q2*4`。
- `np.allclose(a, b, atol=1e-6)`: 2つの配列が許容誤差内で一致するか判定する。グローバル位相の補正後に使用する。

### 3.2 具体例: $|\psi\rangle = |+\rangle$ のテレポーテーション

ノートの具体例と同じ $|+\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$ を転送する。

In [35]:
# |+⟩ = (|0⟩ + |1⟩) / √2
verify_teleportation(1/np.sqrt(2), 1/np.sqrt(2), label="(|+⟩)")

転送する状態: 0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩  (|+⟩)

  Alice の測定: |00⟩
  Bob の状態（補正前）: 0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩
  補正ゲート: I
  Bob の状態（補正後）: 0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩  ✓

  Alice の測定: |01⟩
  Bob の状態（補正前）: 0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩
  補正ゲート: X
  Bob の状態（補正後）: 0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩  ✓

  Alice の測定: |10⟩
  Bob の状態（補正前）: 0.7071+0.0000j|0⟩ + -0.7071+0.0000j|1⟩
  補正ゲート: Z
  Bob の状態（補正後）: 0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩  ✓

  Alice の測定: |11⟩
  Bob の状態（補正前）: -0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩
  補正ゲート: ZX
  Bob の状態（補正後）: 0.7071+0.0000j|0⟩ + 0.7071+0.0000j|1⟩  ✓

  全パターンで |ψ⟩ を復元: 成功


True

### 3.3 さまざまな状態で検証

$|0\rangle$, $|1\rangle$, $|+\rangle$, $|-\rangle$, および一般的な状態で検証する。

In [36]:
test_cases = [
    (1, 0, "|0⟩"),
    (0, 1, "|1⟩"),
    (1/np.sqrt(2), 1/np.sqrt(2), "|+⟩"),
    (1/np.sqrt(2), -1/np.sqrt(2), "|−⟩"),
    (np.cos(np.pi/8), np.sin(np.pi/8), "cos(π/8)|0⟩ + sin(π/8)|1⟩"),
    (1/np.sqrt(3), np.sqrt(2/3)*np.exp(1j*np.pi/4), "複素振幅の状態"),
]

results = []
for alpha, beta, label in test_cases:
    ok = verify_teleportation(alpha, beta, label=label)
    results.append((label, ok))
    print()

print("=" * 50)
print("まとめ:")
for label, ok in results:
    print(f"  {label}: {'成功' if ok else '失敗'}")

転送する状態: 1.0000+0.0000j|0⟩ + 0.0000+0.0000j|1⟩  |0⟩

  Alice の測定: |00⟩
  Bob の状態（補正前）: 1.0000+0.0000j|0⟩ + 0.0000+0.0000j|1⟩
  補正ゲート: I
  Bob の状態（補正後）: 1.0000+0.0000j|0⟩ + 0.0000+0.0000j|1⟩  ✓

  Alice の測定: |01⟩
  Bob の状態（補正前）: 0.0000+0.0000j|0⟩ + 1.0000+0.0000j|1⟩
  補正ゲート: X
  Bob の状態（補正後）: 1.0000+0.0000j|0⟩ + 0.0000+0.0000j|1⟩  ✓

  Alice の測定: |10⟩
  Bob の状態（補正前）: 1.0000+0.0000j|0⟩ + 0.0000+0.0000j|1⟩
  補正ゲート: Z
  Bob の状態（補正後）: 1.0000+0.0000j|0⟩ + 0.0000+0.0000j|1⟩  ✓

  Alice の測定: |11⟩
  Bob の状態（補正前）: 0.0000+0.0000j|0⟩ + 1.0000+0.0000j|1⟩
  補正ゲート: ZX
  Bob の状態（補正後）: 1.0000+0.0000j|0⟩ + 0.0000+0.0000j|1⟩  ✓

  全パターンで |ψ⟩ を復元: 成功

転送する状態: 0.0000+0.0000j|0⟩ + 1.0000+0.0000j|1⟩  |1⟩

  Alice の測定: |00⟩
  Bob の状態（補正前）: 0.0000+0.0000j|0⟩ + 1.0000+0.0000j|1⟩
  補正ゲート: I
  Bob の状態（補正後）: 0.0000+0.0000j|0⟩ + 1.0000+0.0000j|1⟩  ✓

  Alice の測定: |01⟩
  Bob の状態（補正前）: 1.0000+0.0000j|0⟩ + 0.0000+0.0000j|1⟩
  補正ゲート: X
  Bob の状態（補正後）: 0.0000+0.0000j|0⟩ + 1.0000+0.0000j|1⟩  ✓

  Alice の測定: |10⟩
  Bob の状態